# Project 20: Data Quality, Drift & Pipeline Observability

**Simple college project using a large public retail dataset.**

This notebook checks monthly retail data for missing values, duplicates, invalid values, schema changes, data volume changes and drift.

Dataset: **Online Retail transactions** (about 541,909 rows; date range across 2010–2011).

**Before starting:** Download the CSV from Kaggle using the link in the project instructions, then upload it when the next cell asks. The expected file is `Online Retail.csv` (or another CSV name).

## 1. Import libraries
Pandas helps us work with table data, NumPy does calculations, and Matplotlib makes graphs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Libraries ready!")

## 2. Upload the public dataset
Run this cell, choose the CSV you downloaded from Kaggle. The file is large, so upload may take a little time.

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name, encoding="latin1")
print("Dataset loaded!")
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

In [ ]:
df

NameError: name 'df' is not defined

## 3. Understand the data
Each row is a product line on a retail invoice. The dataset contains invoice, product, quantity, date, price, customer and country information.

In [ ]:
df.info()
print("\nMissing values in each column:")
print(df.isnull().sum())
print("\nDataset shape:", df.shape)

## 4. Prepare dates and create monthly snapshots
We turn the invoice date into a real date and split the data into months. These monthly tables are our snapshots.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

months = sorted(df["Month"].unique())
snapshots = {month: df[df["Month"] == month].copy() for month in months}

print("Months in dataset:", months)
print("Number of monthly snapshots:", len(snapshots))
for month, data in snapshots.items():
    print(month, "rows:", len(data))

## 5. Set simple rules
We expect these important columns. We will warn if a column is missing or if a new column appears. The unique key is a combination of invoice, product, and date because one invoice can contain several products.

In [ ]:
EXPECTED_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]

KEY_COLUMNS = ["InvoiceNo", "StockCode", "InvoiceDate"]

NULL_WARNING_PERCENT = 10
DRIFT_WARNING_PERCENT = 25
VOLUME_WARNING_PERCENT = 40

print("Rules are ready")

## 6. Data profile: missing values and data types

In [ ]:
def profile_data(data):
    report = pd.DataFrame({
        "data_type": data.dtypes.astype(str),
        "missing_count": data.isnull().sum(),
        "missing_percent": (data.isnull().mean() * 100).round(2),
        "unique_values": data.nunique()
    })
    return report

profile_data(df)

## 7. Check duplicates
A duplicate means the same invoice/product/date line appears more than once.

In [ ]:
def check_duplicates(data):
    duplicate_count = data.duplicated(subset=KEY_COLUMNS).sum()
    duplicate_percent = duplicate_count / len(data) * 100
    return int(duplicate_count), round(duplicate_percent, 2)

check_duplicates(df)

## 8. Check invalid values
We flag quantities that are zero or negative and unit prices below zero. Negative quantity may also represent returns, so this is a warning to investigate—not automatically proof of bad data.

In [ ]:
def check_invalid_values(data):
    bad_quantity = (data["Quantity"] <= 0).sum()
    bad_price = (data["UnitPrice"] < 0).sum()
    return {
        "quantity_zero_or_negative": int(bad_quantity),
        "negative_unit_price": int(bad_price)
    }

check_invalid_values(df)

## 9. Check schema changes between months

In [ ]:
def compare_schema(old_data, new_data):
    old_columns = set(old_data.columns)
    new_columns = set(new_data.columns)
    return {
        "added_columns": sorted(list(new_columns - old_columns)),
        "missing_columns": sorted(list(old_columns - new_columns))
    }

first_month = snapshots[months[0]]
last_month = snapshots[months[-1]]
compare_schema(first_month, last_month)

NameError: name 'snapshots' is not defined

## 10. Compare monthly data volume

In [ ]:
monthly_counts = df.groupby("Month").size()
volume_report = monthly_counts.to_frame("row_count")
volume_report["change_percent"] = monthly_counts.pct_change() * 100
volume_report["change_percent"] = volume_report["change_percent"].round(2)
volume_report

In [ ]:
plt.figure(figsize=(12, 5))
monthly_counts.plot(kind="bar")
plt.title("Rows in Each Month")
plt.xlabel("Month")
plt.ylabel("Rows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Drift check using NumPy
We compare the average quantity and average unit price of each month with the first month. If the average changes by more than our chosen threshold, we show a warning. This is a simple student-friendly drift check.

In [ ]:
def mean_drift(reference, current):
    ref_mean = np.nanmean(reference)
    current_mean = np.nanmean(current)
    if ref_mean == 0 or np.isnan(ref_mean):
        return 0
    return abs(current_mean - ref_mean) / abs(ref_mean) * 100

reference_month = snapshots[months[0]]
drift_rows = []
for month in months:
    current = snapshots[month]
    for column in ["Quantity", "UnitPrice"]:
        drift = mean_drift(reference_month[column], current[column])
        drift_rows.append({
            "month": month,
            "column": column,
            "reference_mean": round(float(np.nanmean(reference_month[column])), 2),
            "current_mean": round(float(np.nanmean(current[column])), 2),
            "drift_percent": round(drift, 2),
            "alert": drift > DRIFT_WARNING_PERCENT
        })

drift_report = pd.DataFrame(drift_rows)
drift_report

## 12. Simulate a few problems
We make a copy of the latest month and add a few pretend errors (nulls, duplicates, invalid values, an extra column and a changed country label). The original data remains unchanged. This proves our checks can detect problems.

In [ ]:
test_data = last_month.copy()

# Add a few missing values
if len(test_data) > 10:
    test_data.loc[test_data.index[:5], "Country"] = np.nan

# Add duplicate rows
test_data = pd.concat([test_data, test_data.iloc[:10]], ignore_index=True)

# Add invalid quantities
test_data.loc[test_data.index[:5], "Quantity"] = -1

# Simulate a schema change (new column)
test_data["NewFlag"] = "test"

# Simulate a category/label change
test_data["Country"] = test_data["Country"].replace("United Kingdom", "UK")

print("Simulated data shape:", test_data.shape)

NameError: name 'last_month' is not defined

## 13. Run checks on simulated data and create alerts

In [ ]:
def run_checks(data, snapshot_name):
    alerts = []
    # Required columns
    missing_columns = [col for col in EXPECTED_COLUMNS if col not in data.columns]
    extra_columns = [col for col in data.columns if col not in EXPECTED_COLUMNS and col != "Month"]
    if missing_columns:
        alerts.append([snapshot_name, "CRITICAL", "Missing columns", str(missing_columns)])
    if extra_columns:
        alerts.append([snapshot_name, "WARNING", "Extra columns", str(extra_columns)])

    # Missing values
    for col in data.columns:
        missing_percent = data[col].isnull().mean() * 100
        if missing_percent > NULL_WARNING_PERCENT:
            alerts.append([snapshot_name, "WARNING", "High missing values", f"{col}: {missing_percent:.2f}%"] )

    # Duplicate records
    duplicate_count, duplicate_percent = check_duplicates(data)
    if duplicate_count > 0:
        alerts.append([snapshot_name, "WARNING", "Duplicate key rows", f"{duplicate_count} ({duplicate_percent}%)"] )

    # Invalid values
    invalid = check_invalid_values(data)
    for problem, count in invalid.items():
        if count > 0:
            alerts.append([snapshot_name, "WARNING", "Invalid values", f"{problem}: {count}"])

    return alerts

alerts = run_checks(test_data, "SIMULATED_LATEST_MONTH")
alert_log = pd.DataFrame(alerts, columns=["snapshot", "severity", "issue", "details"])
alert_log

## 14. Create a final monitoring summary

In [ ]:
summary = pd.DataFrame({
    "month": months,
    "row_count": [len(snapshots[m]) for m in months],
    "duplicate_rows": [check_duplicates(snapshots[m])[0] for m in months],
    "missing_customer_id": [snapshots[m]["CustomerID"].isnull().sum() for m in months],
    "invalid_quantity": [(snapshots[m]["Quantity"] <= 0).sum() for m in months],
    "negative_price": [(snapshots[m]["UnitPrice"] < 0).sum() for m in months]
})
summary

## 15. Save reports to CSV
These files are your project outputs. Click the download links generated by the next cell.

In [ ]:
summary.to_csv("monthly_quality_summary.csv", index=False)
drift_report.to_csv("drift_report.csv", index=False)
alert_log.to_csv("simulated_alert_log.csv", index=False)

print("Saved 3 report files:")
print("- monthly_quality_summary.csv")
print("- drift_report.csv")
print("- simulated_alert_log.csv")
files.download("monthly_quality_summary.csv")
files.download("drift_report.csv")
files.download("simulated_alert_log.csv")

## 16. Conclusion
This project used a large public retail dataset and split it into monthly snapshots. We checked missing values, duplicate keys, invalid quantities/prices, schema changes, monthly record counts and simple numerical drift. We also simulated data problems and generated a reproducible alert log.

**Limitations:** A mean-only drift check is simple and may miss changes in the shape of a distribution. In a real pipeline, thresholds should be calibrated with historical data and alerts should be reviewed by a data owner.